
# Hindi/Sanskrit Prosody — Syllable Grouping Model

This notebook trains a **sequence-labeling model** to predict 2-mātrā grouping boundaries.

### Intended task
Given a sequence of syllables with features such as:

- syllable identity
- mātrā/weight
- intrinsic vs positional heaviness
- syllable position
- word/verse position

predict a label for each syllable indicating whether it **continues the current group** or **starts a new group**.

The notebook uses a **BIO-style labeling scheme**:

- `B` = beginning of a new 2-mātrā group
- `I` = continuation of the previous group

A group may contain one or more syllables depending on the corpus annotation.

> **Important:** Change the column names in the configuration cell to match your dataset.


In [1]:

# Install dependencies if necessary
# Uncomment when running in Colab/Jupyter for the first time.

!pip install -q pandas numpy scikit-learn joblib


In [2]:

import os
import json
import ast
import numpy as np
import pandas as pd

from collections import Counter, defaultdict

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import joblib

SEED = 42
np.random.seed(SEED)



## 1. Load the dataset

The model expects **one row per syllable**.

If your data is already in that format, set `DATA_PATH` below.

If your dataset instead stores an entire verse/group in one row, adapt the preprocessing cell so that each verse becomes a sequence of syllable records.


In [3]:

# =========================
# CONFIGURATION
# =========================

DATA_PATH = "data.csv"       # <-- change this
FILE_TYPE = "csv"            # "csv" or "json"

# Change these to the actual column names in your dataset.
SEQ_ID_COL = "sequence_id"   # verse/line/example identifier
SYLLABLE_COL = "syllable"
WEIGHT_COL = "weight"        # e.g. 1/2 or laghu/guru
HEAVY_COL = "heaviness"      # e.g. intrinsic / positional / light
POSITION_COL = "position"    # syllable position in sequence

# Ground-truth grouping label.
# Examples: B/I, 0/1, group_id, etc.
LABEL_COL = "label"


def load_data(path, file_type="csv"):
    if file_type.lower() == "csv":
        return pd.read_csv(path)
    elif file_type.lower() == "json":
        return pd.read_json(path)
    else:
        raise ValueError("FILE_TYPE must be 'csv' or 'json'")


df = load_data(DATA_PATH, FILE_TYPE)

print("Shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())


Shape: (40054, 6)


,sequence_id,position,syllable,weight,heaviness,label
0,कंकड़,0,कं,2,intrinsic,B
1,कंकड़,1,क,1,light,B
2,कंकड़,2,ड़,1,light,I
3,कंकण,0,कं,2,intrinsic,B
4,कंकण,1,क,1,light,B



Columns:
['sequence_id', 'position', 'syllable', 'weight', 'heaviness', 'label']


In [4]:

# Inspect the raw data

print(df.info())
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("\nLabel distribution:")
if LABEL_COL in df.columns:
    display(df[LABEL_COL].value_counts(dropna=False))


<class 'pandas.DataFrame'>
RangeIndex: 40054 entries, 0 to 40053
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sequence_id  40054 non-null  str  
 1   position     40054 non-null  int64
 2   syllable     40054 non-null  str  
 3   weight       40054 non-null  int64
 4   heaviness    40054 non-null  str  
 5   label        40054 non-null  str  
dtypes: int64(2), str(4)
memory usage: 1.8 MB
None

Missing values:


sequence_id    0
position       0
syllable       0
weight         0
heaviness      0
label          0
dtype: int64


Label distribution:


label
B    29776
I    10278
Name: count, dtype: int64


## 2. Normalize the labels

This cell converts common label formats into `B` / `I`.

If your dataset already has `B` and `I`, no changes are needed.

If the annotation means something different, modify `normalize_label()`.


In [5]:

def normalize_label(x):
    if pd.isna(x):
        return None

    s = str(x).strip().upper()

    # Common BIO representations
    if s in {"B", "BEGIN", "START", "1", "B-1", "B_GROUP"}:
        return "B"
    if s in {"I", "INSIDE", "CONTINUE", "0", "I-1", "I_GROUP"}:
        return "I"

    return s


df[LABEL_COL] = df[LABEL_COL].map(normalize_label)

print(df[LABEL_COL].value_counts(dropna=False))


label
B    29776
I    10278
Name: count, dtype: int64



## 3. Build sequences

The important part of this problem is that grouping depends on **context**, not only on the weight of the current syllable.

We therefore train on neighboring syllables rather than treating every syllable as an independent sample.

The feature extractor below includes:

- current syllable
- weight
- heaviness
- absolute position
- relative position
- previous/next syllable
- previous/next weight
- previous/next heaviness
- short contextual windows

This is deliberately interpretable and gives us a strong baseline before moving to a neural model.


In [6]:

# Basic cleanup

required = [SEQ_ID_COL, SYLLABLE_COL, WEIGHT_COL, HEAVY_COL, POSITION_COL, LABEL_COL]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"Available columns: {df.columns.tolist()}\n"
        "Update the configuration cell to match your dataset."
    )

df = df.dropna(subset=[SEQ_ID_COL, LABEL_COL]).copy()

# Preserve original order within each sequence.
# If position is numeric, sort by it.
try:
    df[POSITION_COL] = pd.to_numeric(df[POSITION_COL])
    df = df.sort_values([SEQ_ID_COL, POSITION_COL])
except Exception:
    df = df.sort_values([SEQ_ID_COL])

sequences = []

for seq_id, g in df.groupby(SEQ_ID_COL, sort=False):
    rows = g.to_dict("records")
    sequences.append({
        "id": seq_id,
        "rows": rows
    })

print("Number of sequences:", len(sequences))
print("First sequence length:", len(sequences[0]["rows"]) if sequences else 0)


Number of sequences: 13196
First sequence length: 4


In [7]:

def safe_str(x):
    if pd.isna(x):
        return "<UNK>"
    return str(x)


def make_features(rows, i):
    r = rows[i]

    features = {
        "bias": 1,

        # Current syllable
        "syllable": safe_str(r[SYLLABLE_COL]),
        "weight": safe_str(r[WEIGHT_COL]),
        "heaviness": safe_str(r[HEAVY_COL]),

        # Position
        "position": safe_str(r[POSITION_COL]),

        # Useful numerical versions when possible
        "position_numeric": float(r[POSITION_COL])
            if str(r[POSITION_COL]).replace(".", "", 1).isdigit()
            else 0.0,

        # Sequence-relative features
        "is_first": int(i == 0),
        "is_last": int(i == len(rows) - 1),

        # Previous token
        "prev_syllable": safe_str(rows[i-1][SYLLABLE_COL]) if i > 0 else "<START>",
        "prev_weight": safe_str(rows[i-1][WEIGHT_COL]) if i > 0 else "<START>",
        "prev_heaviness": safe_str(rows[i-1][HEAVY_COL]) if i > 0 else "<START>",

        # Next token
        "next_syllable": safe_str(rows[i+1][SYLLABLE_COL]) if i < len(rows)-1 else "<END>",
        "next_weight": safe_str(rows[i+1][WEIGHT_COL]) if i < len(rows)-1 else "<END>",
        "next_heaviness": safe_str(rows[i+1][HEAVY_COL]) if i < len(rows)-1 else "<END>",
    }

    # Wider context
    for offset in [-2, -1, 1, 2]:
        j = i + offset

        if 0 <= j < len(rows):
            prefix = f"offset_{offset}_"
            features[prefix + "syllable"] = safe_str(rows[j][SYLLABLE_COL])
            features[prefix + "weight"] = safe_str(rows[j][WEIGHT_COL])
            features[prefix + "heaviness"] = safe_str(rows[j][HEAVY_COL])
        else:
            prefix = f"offset_{offset}_"
            features[prefix + "syllable"] = "<BOUNDARY>"
            features[prefix + "weight"] = "<BOUNDARY>"
            features[prefix + "heaviness"] = "<BOUNDARY>"

    return features


X_sequences = []
y_sequences = []
group_ids = []

for seq in sequences:
    rows = seq["rows"]

    X_seq = [make_features(rows, i) for i in range(len(rows))]
    y_seq = [normalize_label(r[LABEL_COL]) for r in rows]

    X_sequences.append(X_seq)
    y_sequences.append(y_seq)
    group_ids.append(seq["id"])

print("Example feature dictionary:")
display(pd.DataFrame([X_sequences[0][i] for i in range(min(3, len(X_sequences[0])))]))


Example feature dictionary:


,bias,syllable,weight,heaviness,position,position_numeric,is_first,is_last,prev_syllable,prev_weight,...,offset_-2_heaviness,offset_-1_syllable,offset_-1_weight,offset_-1_heaviness,offset_1_syllable,offset_1_weight,offset_1_heaviness,offset_2_syllable,offset_2_weight,offset_2_heaviness
0,1,कँ,2,intrinsic,0,0.0,1,0,<START>,<START>,...,<BOUNDARY>,<BOUNDARY>,<BOUNDARY>,<BOUNDARY>,ख,1,light,रा,2,intrinsic
1,1,ख,1,light,1,1.0,0,0,कँ,2,...,<BOUNDARY>,कँ,2,intrinsic,रा,2,intrinsic,वी,2,intrinsic
2,1,रा,2,intrinsic,2,2.0,0,0,ख,1,...,intrinsic,ख,1,light,वी,2,intrinsic,<BOUNDARY>,<BOUNDARY>,<BOUNDARY>



## 4. Train/validation/test split

The split is done at the **sequence level**, not the syllable level.

This is important: syllables from the same verse/example must not appear in both training and test sets, otherwise the evaluation can be artificially inflated.


In [8]:

n = len(X_sequences)
indices = np.arange(n)

# 80% train, 10% validation, 10% test
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

n_train = int(0.80 * n)
n_val = int(0.10 * n)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train+n_val]
test_idx = indices[n_train+n_val:]

def flatten(seqs, labels):
    X = [x for seq in seqs for x in seq]
    y = [y for seq in labels for y in seq]
    return X, y

X_train_seq = [X_sequences[i] for i in train_idx]
y_train_seq = [y_sequences[i] for i in train_idx]

X_val_seq = [X_sequences[i] for i in val_idx]
y_val_seq = [y_sequences[i] for i in val_idx]

X_test_seq = [X_sequences[i] for i in test_idx]
y_test_seq = [y_sequences[i] for i in test_idx]

X_train, y_train = flatten(X_train_seq, y_train_seq)
X_val, y_val = flatten(X_val_seq, y_val_seq)
X_test, y_test = flatten(X_test_seq, y_test_seq)

print("Train sequences:", len(train_idx), "syllables:", len(X_train))
print("Val sequences:", len(val_idx), "syllables:", len(X_val))
print("Test sequences:", len(test_idx), "syllables:", len(X_test))


Train sequences: 10556 syllables: 32056
Val sequences: 1319 syllables: 3982
Test sequences: 1321 syllables: 4016



## 5. Train the baseline sequence-labeling model

We use a sparse feature representation + logistic regression as a strong, interpretable baseline.

This is not the final architecture. The purpose is to establish a reliable baseline and determine whether the grouping signal exists in the available features.


In [9]:

vectorizer = DictVectorizer(sparse=True)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("Feature matrix:", X_train_vec.shape)

model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED
)

model.fit(X_train_vec, y_train)

val_pred = model.predict(X_val_vec)
test_pred = model.predict(X_test_vec)

print("Validation accuracy:", accuracy_score(y_val, val_pred))
print("Test accuracy:", accuracy_score(y_test, test_pred))


Feature matrix: (32056, 7772)
Validation accuracy: 0.9713711702661979
Test accuracy: 0.9656374501992032


In [10]:

print("TEST CLASSIFICATION REPORT")
print(classification_report(y_test, test_pred, digits=4))

print("Confusion matrix:")
labels = sorted(set(y_test) | set(test_pred))
display(pd.DataFrame(
    confusion_matrix(y_test, test_pred, labels=labels),
    index=labels,
    columns=labels
))


TEST CLASSIFICATION REPORT
              precision    recall  f1-score   support

           B     0.9917    0.9617    0.9765      2980
           I     0.8988    0.9768    0.9362      1036

    accuracy                         0.9656      4016
   macro avg     0.9452    0.9693    0.9563      4016
weighted avg     0.9677    0.9656    0.9661      4016

Confusion matrix:


,B,I
B,2866,114
I,24,1012



## 6. Evaluate at the sequence/grouping level

Per-syllable accuracy is useful, but the actual task is **grouping**.

A single incorrect boundary can change the entire downstream grouping, so we also reconstruct predicted groups and compare them with the gold groups.


In [11]:

def groups_from_labels(rows, labels):
    groups = []
    current = []

    for row, label in zip(rows, labels):
        if label == "B" and current:
            groups.append(current)
            current = []

        current.append(safe_str(row[SYLLABLE_COL]))

    if current:
        groups.append(current)

    return groups


# Reconstruct test predictions sequence by sequence
test_sequence_results = []

for idx in test_idx:
    rows = sequences[idx]["rows"]

    X_seq = [make_features(rows, i) for i in range(len(rows))]
    pred_seq = model.predict(vectorizer.transform(X_seq))

    gold_seq = [normalize_label(r[LABEL_COL]) for r in rows]

    gold_groups = groups_from_labels(rows, gold_seq)
    pred_groups = groups_from_labels(rows, pred_seq)

    test_sequence_results.append({
        "sequence_id": sequences[idx]["id"],
        "gold_groups": gold_groups,
        "pred_groups": pred_groups,
        "gold_labels": gold_seq,
        "pred_labels": list(pred_seq)
    })


exact_sequence_match = np.mean([
    x["gold_groups"] == x["pred_groups"]
    for x in test_sequence_results
])

print("Exact grouping accuracy:", exact_sequence_match)


Exact grouping accuracy: 0.9038607115821348


In [12]:

# Show examples where the model disagrees with the gold annotation

errors = [
    x for x in test_sequence_results
    if x["gold_groups"] != x["pred_groups"]
]

print("Incorrect sequences:", len(errors), "/", len(test_sequence_results))

for x in errors[:10]:
    print("\nSequence:", x["sequence_id"])
    print("GOLD:", " | ".join(" ".join(g) for g in x["gold_groups"]))
    print("PRED:", " | ".join(" ".join(g) for g in x["pred_groups"]))


Incorrect sequences: 127 / 1321

Sequence: फ़ैक्टरी
GOLD: फ़ै | क्ट | री
PRED: फ़ै क्ट | री

Sequence: सटकना
GOLD: स ट | क | ना
PRED: स | ट क | ना

Sequence: विपत्ति
GOLD: वि | प | त्ति
PRED: वि | प त्ति

Sequence: शहसवार
GOLD: श | ह स | वा र
PRED: श ह स | वा र

Sequence: सारना
GOLD: सा | र | ना
PRED: सा र | ना

Sequence: क्रियाविधि
GOLD: क्रि | या | वि | धि
PRED: क्रि | या | वि धि

Sequence: तदनुरूप
GOLD: त | द | नु | रू प
PRED: त द | नु | रू प

Sequence: दुर्व्यवहार
GOLD: दु र्व्य | व | हा र
PRED: दु | र्व्य व | हा र

Sequence: गडमड
GOLD: ग ड | म ड
PRED: ग | ड | म ड

Sequence: पसली
GOLD: प | स | ली
PRED: प स | ली



## 7. Inspect model decisions

The next cell lets us inspect the model's predicted probability for each syllable.

This is particularly useful for finding examples of the phenomenon we observed:

> A guru syllable sometimes absorbs the following laghu and sometimes does not.

Those examples should be inspected directly rather than forcing a hand-written phonetic rule.


In [13]:

def inspect_sequence(sequence_index):
    rows = sequences[sequence_index]["rows"]

    X_seq = [make_features(rows, i) for i in range(len(rows))]
    X_vec = vectorizer.transform(X_seq)

    pred = model.predict(X_vec)
    probs = model.predict_proba(X_vec)

    output = []

    for i, row in enumerate(rows):
        p = dict(zip(model.classes_, probs[i]))

        output.append({
            "position": row[POSITION_COL],
            "syllable": row[SYLLABLE_COL],
            "weight": row[WEIGHT_COL],
            "heaviness": row[HEAVY_COL],
            "gold": normalize_label(row[LABEL_COL]),
            "prediction": pred[i],
            "P(B)": p.get("B", 0),
            "P(I)": p.get("I", 0),
        })

    return pd.DataFrame(output)


# Example:
# display(inspect_sequence(test_idx[0]))



## 8. Save the trained model

The vectorizer and classifier are saved together so that the exact feature mapping used during training can be reused for inference.


In [ ]:

os.makedirs("trained_model", exist_ok=True)

joblib.dump(
    {
        "vectorizer": vectorizer,
        "model": model,
        "config": {
            "SEQ_ID_COL": SEQ_ID_COL,
            "SYLLABLE_COL": SYLLABLE_COL,
            "WEIGHT_COL": WEIGHT_COL,
            "HEAVY_COL": HEAVY_COL,
            "POSITION_COL": POSITION_COL,
            "LABEL_COL": LABEL_COL,
        }
    },
    "trained_model/syllable_grouping_model.joblib"
)

print("Saved to trained_model/syllable_grouping_model.joblib")



## 9. Inference on a new sequence

Provide a list of syllable dictionaries with the same feature fields used during training.


In [15]:

def predict_sequence(rows):
    X = [make_features(rows, i) for i in range(len(rows))]
    X_vec = vectorizer.transform(X)

    labels = model.predict(X_vec)
    probabilities = model.predict_proba(X_vec)

    result = []

    for row, label, prob in zip(rows, labels, probabilities):
        result.append({
            "syllable": row[SYLLABLE_COL],
            "weight": row[WEIGHT_COL],
            "heaviness": row[HEAVY_COL],
            "prediction": label,
            "confidence": float(np.max(prob))
        })

    return pd.DataFrame(result)


# Example:
#
new_sequence = [
    {
        SYLLABLE_COL: "का",
        WEIGHT_COL: 2,
        HEAVY_COL: "intrinsic",
        POSITION_COL: 0,
    },
    {
        SYLLABLE_COL: "वि",
        WEIGHT_COL: 1,
        HEAVY_COL: "light",
        POSITION_COL: 1,
    },
]

display(predict_sequence(new_sequence))


,syllable,weight,heaviness,prediction,confidence
0,का,2,intrinsic,B,1.000000
1,वि,1,light,I,0.557709



# Next step: stronger model

If this baseline confirms that the dataset contains useful contextual information, the next model I would test is a **BiLSTM/Transformer sequence labeler**.

Useful additions:

1. syllable embedding
2. weight embedding
3. heaviness embedding
4. relative position
5. word boundary
6. neighboring syllables
7. word/compound identity
8. previous predicted label
9. attention over the full verse

For the final evaluation, keep **exact sequence/grouping accuracy** alongside ordinary token-level accuracy. The latter alone can hide boundary errors.
